<a href="https://colab.research.google.com/github/mekdes10-30/Ev-parts-Task/blob/main/notebook_review_copy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV Spare-Parts Import Prioritization — Research, Cleaning & Analysis Notebook
**Individual internship assignment | Ethiopian market context**

This notebook documents the project from assignment scope through raw-data audit and cleaning. It is designed to be understandable and reproducible.

**Project question:** Which five EV replacement-part categories should a prospective importer prioritize for Ethiopia, given incomplete public evidence?

**Workflow:** assignment/scope → source and collection methodology → load raw files → inspect and validate → assess evidence quality and coverage → clean/standardize → validate cleaning → export cleaned data.

> This notebook does not assume that online listing counts equal Ethiopian registrations, sales, demand, or stock. It does not treat missing information as zero or confirmed unavailability. Scoring and recommendations are intentionally left for a later stage after the data is reviewed and a defensible method is agreed.

## 1. Assignment requirements and our decisions

### Trainer's assignment (PPT)
The presentation describes five connected tasks:
1. Select **three** vehicles from the candidate list.
2. Investigate ten assigned part categories.
3. Build and clean a dataset from public online evidence.
4. Design an explainable scoring method.
5. Recommend five high-priority parts.

It also asks for source URLs and collection dates, two sources where possible, traceable compatibility/price/availability evidence, transparent missing values and assumptions, a justified scoring formula, weight sensitivity, raw and cleaned data, source log, analysis notebook, recommendations, and limitations.

### Our documented scope decision
We deliberately selected **four vehicles**, rather than the PPT's three:
- BYD Seagull
- BYD Yuan Up
- Changan BenBen E-Star
- Toyota bZ4X

This is our project decision—not a PPT requirement. We must explain it transparently in our presentation and not imply that the trainer required four.

### Ten target categories (PPT)
1. Brake pads
2. Brake discs
3. Cabin air filters
4. Wiper blades
5. Headlights
6. Tail lights
7. Side mirrors
8. Bumpers
9. Fenders/body panels
10. Shock absorbers

The PPT allows a category replacement if evidence is unavailable, provided the replacement is explained. We have retained the ten categories above.

## 2. Research objective and interpretation

The project has two evidence dimensions:
- **International sourcing evidence:** part identification, supplier listings, price indications, compatibility claims and sourcing options.
- **Ethiopian market evidence:** local vehicle presence, local part availability/prices, supply gaps, and demand proxies.

International listings help investigate sourcing; they do **not** prove Ethiopian demand or local scarcity. A vehicle listing is not a registration or sales count. A supplier's capability statement is not an item-level stock confirmation.

## 3. Research methodology and sources

### Collection approach
The raw dataset was collected in two phases:
- Phase 1: initial public-source research and records.
- Phase 2: supplementary research to improve coverage, especially BYD Yuan Up and Changan BenBen E-Star.

Research included web search excerpts, some fetched page text, supplier/catalogue pages, marketplaces, Chinese-language queries, and Ethiopian/Amharic context searches. Each observed listing or evidence observation is represented separately and linked to an evidence-log row.

### Trainer's common source pack (PPT slide 6)
- Adey Market EV listings — Ethiopian vehicle visibility
- Gezat Electric model catalogue — Ethiopian EV/model context
- Landin BYD parts — international BYD parts/fitment/price reference
- CQSW BYD Seagull part-number list — international parts reference
- Toyota bZ4X catalogue — international catalogue reference
- NHTSA datasets/APIs — technical/vehicle information (not a local market source)

The source pack is mixed: **not all sources are Ethiopian**. The PPT says to record every source URL and collection date and use at least two sources where possible. This means corroboration when possible, not inventing a second source.

### Source-quality principles from PPT slide 8
- Prefer identifiable manufacturers and established catalogues.
- Cross-check more than one source where possible.
- Standardize currencies, units and vehicle names.
- Record unavailable values rather than inventing them.
- Separate observed facts from analyst judgement.

## 4. Our dataset design

The raw dataset contains 16 fields. The evidence log preserves observation-level provenance. We retain these schemas rather than silently changing the collection structure.

| Field | Why it is useful |
|---|---|
| record_id | Unique raw observation identifier |
| vehicle | Target vehicle |
| manufacturer | Vehicle/part manufacturer as recorded |
| part_category | One of the ten target categories (or contextual N/A row) |
| source_name | Source identity |
| source_type | Source class |
| source_url | Traceable source location |
| collection_date | When evidence was collected |
| evidence_type | Kind of observation |
| compatibility_status | Compatibility confidence/claim |
| compatibility_evidence | Basis for compatibility statement |
| part_number | Listing/catalogue part number |
| price | Observed price text/value |
| currency | Currency attached to price |
| availability_status | Availability claim as recorded |
| oe_part_number | Original-equipment part number if observed |

The evidence log records evidence ID, linked record ID, source, URL/date, observed information, page/product identifier, collection method, verification status, and access limitations.

The PPT requires that the dataset identify vehicle/part, source/date, compatibility evidence, price/availability evidence, and basis for recommendation. Our fields support these needs, although not every record contains every value.

## 5. Upload and extract the collected raw files

Upload the Phase 2 ZIP containing:
- `EV_Parts_Raw_Dataset.csv`
- `EV_Parts_Evidence_Log.csv`
- `EV_Parts_Coverage_Matrix.csv`
- `EV_Parts_Research_Notes.md`

The ZIP is used as input; original CSVs are kept unchanged.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving files (1).zip to files (1).zip


In [ ]:
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import csv
import re
import unicodedata
from IPython.display import display

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Upload the research ZIP file.")

BASE_DIR = Path("/content/ev_parts_project")
RAW_DIR = BASE_DIR / "raw_original"
OUTPUT_DIR = BASE_DIR / "output"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(RAW_DIR)

for path in RAW_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(RAW_DIR))

EV_Parts_Coverage_Matrix.csv
EV_Parts_Research_Notes.md
EV_Parts_Raw_Dataset.csv
EV_Parts_Evidence_Log.csv


## 6. Load original files

Read every column as text initially. This prevents pandas from automatically converting identifiers, part numbers, dates, or strings that resemble numbers.

In [ ]:
raw_file = next(RAW_DIR.rglob("EV_Parts_Raw_Dataset.csv"))
evidence_file = next(RAW_DIR.rglob("EV_Parts_Evidence_Log.csv"))
coverage_file = next(RAW_DIR.rglob("EV_Parts_Coverage_Matrix.csv"))
notes_file = next(RAW_DIR.rglob("EV_Parts_Research_Notes.md"))

raw_original = pd.read_csv(raw_file, encoding="utf-8", dtype=str, keep_default_na=False)
evidence_original = pd.read_csv(evidence_file, encoding="utf-8", dtype=str, keep_default_na=False)
coverage_original = pd.read_csv(coverage_file, encoding="utf-8", dtype=str, keep_default_na=False)

# Separate working copies. Do not modify *_original DataFrames.
raw = raw_original.copy(deep=True)
evidence = evidence_original.copy(deep=True)
coverage = coverage_original.copy(deep=True)

print("Raw:", raw.shape, "| Evidence:", evidence.shape, "| Coverage:", coverage.shape)
display(raw.head(5))

Raw: (197, 16) | Evidence: (197, 10) | Coverage: (40, 7)


,record_id,vehicle,manufacturer,part_category,source_name,source_type,source_url,collection_date,evidence_type,compatibility_status,compatibility_evidence,part_number,price,currency,availability_status,oe_part_number
0,REC-0001,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1,¥64,RMB,,EQEA-3501500CH
1,REC-0002,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1,¥64,RMB,,EQEA-3502140SS
2,REC-0003,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260F1P,¥64,RMB,,EQEA-3501500CH 14758996-00
3,REC-0004,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake pads,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Pads BYD0042260...",BYD0042260R1P,¥64,RMB,,EQEA-3502140SS 14753311-00
4,REC-0005,BYD Seagull,BYD (vehicle maker); part manufacturer not sta...,Brake discs,Landin Parts (landinparts.com),aftermarket body/parts supplier catalogue (China),https://www.landinparts.com/p.html#/byd/byd004...,2026-11-15,catalogue/product listing (price listing where...,supplier_stated_fitment_unverified,"Displayed listing text: ""Brake Disc BYD0041960...",BYD0041960F1,¥108,RMB,,EM2E-3501011 13423467-00


## 7. Raw dataset inspection

Inspect dimensions, field names, data types, unique values, and blank strings. Blank strings are intentional representations of unavailable values in this collection.

In [ ]:
print("Raw columns:")
print(raw.columns.tolist())
print("\nRaw dimensions:", raw.shape)
print("\nEvidence dimensions:", evidence.shape)
print("\nCoverage dimensions:", coverage.shape)

profile = pd.DataFrame({
    "column": raw.columns,
    "nonblank": [(raw[c].astype(str).str.strip() != "").sum() for c in raw.columns],
    "blank": [(raw[c].astype(str).str.strip() == "").sum() for c in raw.columns],
    "unique_values": [raw[c].nunique(dropna=False) for c in raw.columns]
})
display(profile)

Raw columns:
['record_id', 'vehicle', 'manufacturer', 'part_category', 'source_name', 'source_type', 'source_url', 'collection_date', 'evidence_type', 'compatibility_status', 'compatibility_evidence', 'part_number', 'price', 'currency', 'availability_status', 'oe_part_number']

Raw dimensions: (197, 16)

Evidence dimensions: (197, 10)

Coverage dimensions: (40, 7)


,column,nonblank,blank,unique_values
0,record_id,197,0,197
1,vehicle,197,0,5
2,manufacturer,197,0,17
3,part_category,197,0,13
4,source_name,197,0,56
5,source_type,197,0,38
6,source_url,197,0,128
7,collection_date,197,0,1
8,evidence_type,197,0,28
9,compatibility_status,197,0,13


## 8. Structural validation

Check the schema, identifier uniqueness, exact duplicate rows, and the one-to-one raw/evidence linkage. Any findings are displayed for review; this section does not delete records.

In [ ]:
EXPECTED_RAW = [
    "record_id","vehicle","manufacturer","part_category","source_name","source_type",
    "source_url","collection_date","evidence_type","compatibility_status",
    "compatibility_evidence","part_number","price","currency","availability_status",
    "oe_part_number"
]
EXPECTED_EVIDENCE = [
    "evidence_id","record_id","source_name","source_url","collection_date",
    "observed_information","product_or_page_identifier","collection_method",
    "verification_status","access_limitations"
]

print("Missing raw columns:", sorted(set(EXPECTED_RAW)-set(raw.columns)))
print("Unexpected raw columns:", sorted(set(raw.columns)-set(EXPECTED_RAW)))
print("Missing evidence columns:", sorted(set(EXPECTED_EVIDENCE)-set(evidence.columns)))
print("Unexpected evidence columns:", sorted(set(evidence.columns)-set(EXPECTED_EVIDENCE)))

print("Raw row count:",len(raw),"unique record IDs:",raw.record_id.nunique())
print("Evidence row count:",len(evidence),"unique evidence IDs:",evidence.evidence_id.nunique())
print("Exact duplicate raw rows:",raw.duplicated().sum())
print("Exact duplicate evidence rows:",evidence.duplicated().sum())

raw_ids=set(raw.record_id)
ev_ids=set(evidence.record_id)
print("Raw IDs with no evidence:",len(raw_ids-ev_ids))
print("Evidence IDs not in raw:",len(ev_ids-raw_ids))

# Detect multiple evidence rows per raw record (expected one in this collection).
display(evidence.groupby("record_id").size().loc[lambda s:s!=1].rename("evidence_rows").to_frame())

Missing raw columns: []
Unexpected raw columns: []
Missing evidence columns: []
Unexpected evidence columns: []
Raw row count: 197 unique record IDs: 197
Evidence row count: 197 unique evidence IDs: 197
Exact duplicate raw rows: 0
Exact duplicate evidence rows: 0
Raw IDs with no evidence: 0
Evidence IDs not in raw: 0


,evidence_rows
record_id,


## 9. Coverage analysis (40 vehicle/category cells)

Coverage means that at least one observation was collected for a vehicle/category combination. It does **not** mean that compatibility, stock, demand, or evidence quality is confirmed.

The supplied matrix distinguishes model-named evidence from weak evidence and no evidence. We inspect it and independently derive the combinations represented in raw records.

In [ ]:
display(coverage_original)

part_rows = raw[~raw["part_category"].str.strip().str.upper().str.startswith("N/A")].copy()
observed = (part_rows.groupby(["vehicle","part_category"]).size()
            .rename("raw_record_count").reset_index())
display(observed.sort_values(["vehicle","part_category"]))

print("Distinct vehicle/category pairs in raw part rows:",len(set(zip(observed.vehicle,observed.part_category))))

,vehicle,part_category,records_found,sources_investigated,evidence_found,research_limitations,additional_verification_required
0,BYD Seagull,Brake pads,8,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 8 from 4 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
1,BYD Seagull,Brake discs,3,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 3 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
2,BYD Seagull,Cabin air filters,6,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 6 from 5 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
3,BYD Seagull,Wiper blades,3,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 3 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
4,BYD Seagull,Headlights,5,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 5 from 3 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
5,BYD Seagull,Tail lights,4,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 4 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
6,BYD Seagull,Side mirrors,2,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 2 from 1 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...
7,BYD Seagull,Bumpers,10,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 10 from 2 source name(s); status mix:...,Open Landin/JLL product detail pages; verify O...
8,BYD Seagull,Fenders/body panels,11,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 11 from 2 source name(s); status mix:...,Open Landin/JLL product detail pages; verify O...
9,BYD Seagull,Shock absorbers,7,"Phase 1: Landin Parts (fetched), CQSW Seagull ...",Yes - model-named listing/catalogue entry,Records: 7 from 2 source name(s); status mix: ...,Open Landin/JLL product detail pages; verify O...


,vehicle,part_category,raw_record_count
0,BYD Seagull,Brake discs,3
1,BYD Seagull,Brake pads,8
2,BYD Seagull,Bumpers,10
3,BYD Seagull,Cabin air filters,6
4,BYD Seagull,Fenders/body panels,11
5,BYD Seagull,Headlights,5
6,BYD Seagull,Shock absorbers,7
7,BYD Seagull,Side mirrors,2
8,BYD Seagull,Tail lights,4
9,BYD Seagull,Wiper blades,3


Distinct vehicle/category pairs in raw part rows: 40


## 10. Evidence-log and provenance checks

The PPT asks for URLs and collection dates and cross-checking where possible. This audit checks whether provenance fields are populated and whether raw/evidence source details agree. A URL being present does not itself prove that a page was fully opened or independently verified.

In [ ]:
# Join the evidence log to raw records for consistency checks.
audit = raw_original[["record_id","source_name","source_url","collection_date"]].merge(
    evidence_original[["record_id","source_name","source_url","collection_date","verification_status","access_limitations"]],
    on="record_id",suffixes=("_raw","_evidence"),how="outer",indicator=True
)
print(audit["_merge"].value_counts())
for field in ["source_name","source_url","collection_date"]:
    mismatch = audit[f"{field}_raw"] != audit[f"{field}_evidence"]
    print(field,"mismatches:",mismatch.sum())
display(audit[audit["_merge"]!="both"])
display(audit[(audit["source_url_raw"]!=audit["source_url_evidence"])])

_merge
both          197
left_only       0
right_only      0
Name: count, dtype: int64
source_name mismatches: 0
source_url mismatches: 0
collection_date mismatches: 0


,record_id,source_name_raw,source_url_raw,collection_date_raw,source_name_evidence,source_url_evidence,collection_date_evidence,verification_status,access_limitations,_merge


,record_id,source_name_raw,source_url_raw,collection_date_raw,source_name_evidence,source_url_evidence,collection_date_evidence,verification_status,access_limitations,_merge


## 10.1 Independent source cross-checking



This section compares records belonging to the same vehicle and part category.

We examine:
- Whether different sources report the same part number or OE number.
- Whether the sources represent genuinely different suppliers or related websites.
- Whether compatibility claims agree or contain uncertainty.
- Whether the evidence was obtained from an opened page or only a search-result snippet.

Known related sources are grouped into the same source family to avoid counting them as independent confirmations.

**Important:** Matching part numbers indicate potential corroboration of part identity. They do not independently prove vehicle compatibility, model-year fitment, availability or orderability. All candidate matches remain subject to manual review.

In [ ]:
# PART 10.1 — Corrected independent source cross-checking

import re
import pandas as pd
from itertools import combinations
from IPython.display import display

# ---------------------------------------------------------
# 1. Prepare raw records and evidence log
# ---------------------------------------------------------

required_raw = [
    "record_id", "vehicle", "part_category",
    "source_name", "source_url",
    "part_number", "oe_part_number"
]

required_evidence = [
    "record_id", "verification_status",
    "collection_method", "access_limitations"
]

missing_raw = [c for c in required_raw if c not in raw_original.columns]
missing_evidence = [c for c in required_evidence if c not in evidence_original.columns]

if missing_raw or missing_evidence:
    raise ValueError(
        f"Missing raw columns: {missing_raw}\n"
        f"Missing evidence columns: {missing_evidence}"
    )

crosscheck_data = raw_original[required_raw].copy()

# Rename evidence columns to prevent collisions during merging.
evidence_for_crosscheck = evidence_original[required_evidence].rename(
    columns={
        "verification_status": "evidence_verification_status",
        "collection_method": "evidence_collection_method",
        "access_limitations": "evidence_access_limitations"
    }
)

crosscheck_data = crosscheck_data.merge(
    evidence_for_crosscheck,
    on="record_id",
    how="left",
    validate="one_to_one"
)

# ---------------------------------------------------------
# 2. Identify source families
# ---------------------------------------------------------

def identify_source_family(source_name):
    name = str(source_name).lower()

    # Known related Toyota retailer websites
    if "toyotapartsnow" in name or "toyotapartsdeal" in name:
        return "ToyotaPartsNow/ToyotaPartsDeal"

    # Other known source groups
    if "landin" in name:
        return "Landin"

    if "jll" in name:
        return "JLL Auto Parts"

    if "cqsw" in name or "chongqing shiwei" in name:
        return "Chongqing Shiwei"

    if "aliexpress" in name:
        return "AliExpress"

    if "amazon" in name:
        return "Amazon"

    if "byd-accessories.pages.dev" in name:
        return "BYD Accessories Aggregator"

    if "directfactoryparts" in name:
        return "DirectFactoryParts"

    if "neozero" in name:
        return "NeoZeroAuto"

    if "north park toyota" in name:
        return "North Park Toyota"

    if "nucar" in name:
        return "Nucar Toyota"

    if "conquest" in name:
        return "Conquest Auto"

    if "nunawading" in name:
        return "Nunawading Toyota"

    # Alibaba sellers are kept separate by their recorded source name.
    if "alibaba" in name:
        return str(source_name).strip()

    return str(source_name).strip()


crosscheck_data["source_family"] = (
    crosscheck_data["source_name"].apply(identify_source_family)
)

# ---------------------------------------------------------
# 3. Classify the type of evidence actually collected
# ---------------------------------------------------------

def classify_evidence(row):
    status = str(row["evidence_verification_status"]).lower()
    method = str(row["evidence_collection_method"]).lower()
    source = str(row["source_name"]).lower()

    if "snippet" in status or "web_search" in method:
        return "Search-result snippet"

    if "web_fetch" in method and "landin" in source:
        return "Fetched Landin catalogue listing"

    if "web_fetch" in method:
        return "Fetched page/listing text"

    return "Other / unclear collection method"


crosscheck_data["evidence_type"] = crosscheck_data.apply(
    classify_evidence, axis=1
)

# ---------------------------------------------------------
# 4. Extract identifiers from structured fields only
# ---------------------------------------------------------

def extract_identifiers(value):
    """
    Extract identifier-like tokens from structured part-number fields.
    Does not extract numbers from product descriptions or titles.
    """
    if pd.isna(value):
        return set()

    value = str(value).upper().strip()

    if not value:
        return set()

    # Split fields that contain multiple identifiers.
    pieces = re.split(r"[,;/|\n]+", value)

    identifiers = set()

    for piece in pieces:
        piece = piece.strip()

        # Keep reasonably long alphanumeric identifiers.
        tokens = re.findall(r"[A-Z0-9][A-Z0-9-]{4,}", piece)

        for token in tokens:
            normalized = re.sub(r"[^A-Z0-9]", "", token)

            if len(normalized) >= 5:
                identifiers.add(normalized)

    return identifiers


crosscheck_data["identifiers"] = crosscheck_data.apply(
    lambda row: (
        extract_identifiers(row["part_number"])
        | extract_identifiers(row["oe_part_number"])
    ),
    axis=1
)

# ---------------------------------------------------------
# 5. Compare records within the same vehicle/category
# ---------------------------------------------------------

candidate_rows = []

part_records = crosscheck_data[
    crosscheck_data["part_category"].astype(str).str.strip().ne("")
    & ~crosscheck_data["part_category"].astype(str).str.upper().str.startswith("N/A")
].copy()

for (vehicle, category), group in part_records.groupby(
    ["vehicle", "part_category"], dropna=False
):

    records = group.to_dict("records")

    for record_a, record_b in combinations(records, 2):

        shared_identifiers = (
            record_a["identifiers"] & record_b["identifiers"]
        )

        if not shared_identifiers:
            continue

        for identifier in sorted(shared_identifiers):

            family_a = record_a["source_family"]
            family_b = record_b["source_family"]

            same_family = family_a == family_b

            candidate_rows.append({
                "vehicle": vehicle,
                "part_category": category,
                "matching_identifier": identifier,

                "record_id_1": record_a["record_id"],
                "source_name_1": record_a["source_name"],
                "source_family_1": family_a,
                "evidence_type_1": record_a["evidence_type"],
                "verification_status_1": record_a["evidence_verification_status"],

                "record_id_2": record_b["record_id"],
                "source_name_2": record_b["source_name"],
                "source_family_2": family_b,
                "evidence_type_2": record_b["evidence_type"],
                "verification_status_2": record_b["evidence_verification_status"],

                "same_source_family": same_family,

                "source_relationship_assessment": (
                    "Same identified source family"
                    if same_family
                    else "Potentially distinct source families — independence not established"
                )
            })


crosscheck_matches = pd.DataFrame(candidate_rows)

# ---------------------------------------------------------
# 6. Build a corrected summary
# ---------------------------------------------------------

if not crosscheck_matches.empty:

    summary_rows = []

    for (vehicle, category), group in crosscheck_matches.groupby(
        ["vehicle", "part_category"]
    ):

        record_ids = set(
            group["record_id_1"].tolist()
            + group["record_id_2"].tolist()
        )

        involved_records = crosscheck_data[
            crosscheck_data["record_id"].isin(record_ids)
        ]

        distinct_families = set(
            group["source_family_1"].tolist()
            + group["source_family_2"].tolist()
        )

        independent_candidate_rows = group[
            group["same_source_family"] == False
        ]

        distinct_identifiers = group[
            "matching_identifier"
        ].nunique()

        summary_rows.append({
            "vehicle": vehicle,
            "part_category": category,

            "records_involved": len(record_ids),

            "distinct_source_names": involved_records["source_name"].nunique(),

            "distinct_source_families": len(distinct_families),

            "search_snippet_records": (
                involved_records["evidence_type"] == "Search-result snippet"
            ).sum(),

            "fetched_catalogue_records": (
                involved_records["evidence_type"] == "Fetched Landin catalogue listing"
            ).sum(),

            "other_fetched_page_records": (
                involved_records["evidence_type"] == "Fetched page/listing text"
            ).sum(),

            "pairwise_match_rows": len(group),

            "potentially_distinct_family_pair_rows": len(
                independent_candidate_rows
            ),

            "unique_matching_identifiers": distinct_identifiers,

            "review_status": (
                "Manual verification required — candidate matches only"
            )
        })

    crosscheck_summary = pd.DataFrame(summary_rows)

else:
    crosscheck_summary = pd.DataFrame()

# ---------------------------------------------------------
# 7. Display results
# ---------------------------------------------------------

print("RAW RECORDS:", len(raw_original))
print("EVIDENCE RECORDS:", len(evidence_original))
print("PAIRWISE IDENTIFIER MATCH CANDIDATES:", len(crosscheck_matches))

if not crosscheck_matches.empty:
    print(
        "POTENTIALLY DISTINCT SOURCE-FAMILY PAIR ROWS:",
        crosscheck_matches[
            crosscheck_matches["same_source_family"] == False
        ].shape[0]
    )

print("\nCROSS-CHECK SUMMARY")
display(crosscheck_summary)

print("\nCANDIDATE MATCH DETAILS")
display(crosscheck_matches)

# ---------------------------------------------------------
# 8. Export reports
# ---------------------------------------------------------

summary_path = OUTPUT_DIR / "EV_Parts_Crosscheck_Summary.csv"
matches_path = OUTPUT_DIR / "EV_Parts_Crosscheck_Candidate_Matches.csv"

crosscheck_summary.to_csv(summary_path, index=False)
crosscheck_matches.to_csv(matches_path, index=False)

print("\nExported:")
print(summary_path)
print(matches_path)

RAW RECORDS: 197
EVIDENCE RECORDS: 197
PAIRWISE IDENTIFIER MATCH CANDIDATES: 42
POTENTIALLY DISTINCT SOURCE-FAMILY PAIR ROWS: 20

CROSS-CHECK SUMMARY


,vehicle,part_category,records_involved,distinct_source_names,distinct_source_families,search_snippet_records,fetched_catalogue_records,other_fetched_page_records,pairwise_match_rows,potentially_distinct_family_pair_rows,unique_matching_identifiers,review_status
0,BYD Seagull,Brake pads,4,1,1,0,4,0,2,0,2,Manual verification required — candidate matches only
1,BYD Seagull,Bumpers,2,1,1,0,2,0,1,0,1,Manual verification required — candidate matches only
2,BYD Seagull,Cabin air filters,3,3,3,3,0,0,4,4,2,Manual verification required — candidate matches only
3,BYD Seagull,Headlights,2,2,2,1,1,0,1,1,1,Manual verification required — candidate matches only
4,BYD Seagull,Shock absorbers,5,1,1,0,5,0,4,0,2,Manual verification required — candidate matches only
5,BYD Yuan Up,Tail lights,3,3,3,3,0,0,2,2,2,Manual verification required — candidate matches only
6,Toyota bZ4X,Brake discs,2,2,1,2,0,0,1,0,1,Manual verification required — candidate matches only
7,Toyota bZ4X,Brake pads,8,4,3,8,0,0,13,7,2,Manual verification required — candidate matches only
8,Toyota bZ4X,Bumpers,8,3,2,8,0,0,4,1,4,Manual verification required — candidate matches only
9,Toyota bZ4X,Cabin air filters,6,4,3,6,0,0,6,4,2,Manual verification required — candidate matches only



CANDIDATE MATCH DETAILS


,vehicle,part_category,matching_identifier,record_id_1,source_name_1,source_family_1,evidence_type_1,verification_status_1,record_id_2,source_name_2,source_family_2,evidence_type_2,verification_status_2,same_source_family,source_relationship_assessment
0,BYD Seagull,Brake pads,EQEA3501500CH,REC-0001,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,REC-0003,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,True,Same identified source family
1,BYD Seagull,Brake pads,EQEA3502140SS,REC-0002,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,REC-0004,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,True,Same identified source family
2,BYD Seagull,Bumpers,EQEA2804110A,REC-0020,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,REC-0021,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,True,Same identified source family
3,BYD Seagull,Cabin air filters,1476508400,REC-0051,byd-accessories.pages.dev ('Best BYD spare parts' aggregator),BYD Accessories Aggregator,Search-result snippet,observed_in_search_snippet_page_not_opened; part_number_field_inferred_from_title_suffix,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — independence not established
4,BYD Seagull,Cabin air filters,1476508400,REC-0051,byd-accessories.pages.dev ('Best BYD spare parts' aggregator),BYD Accessories Aggregator,Search-result snippet,observed_in_search_snippet_page_not_opened; part_number_field_inferred_from_title_suffix,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — independence not established
5,BYD Seagull,Cabin air filters,1476508400,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — independence not established
6,BYD Seagull,Cabin air filters,EQEA8100242,REC-0190,AliExpress item 3256807799561527,AliExpress,Search-result snippet,observed_in_search_snippet_page_not_opened,REC-0191,Amazon.com search results 'seagull filter',Amazon,Search-result snippet,observed_in_search_snippet_page_not_opened,False,Potentially distinct source families — independence not established
7,BYD Seagull,Headlights,EQEA4121010B,REC-0009,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,REC-0196,JLL Auto Parts - BYD Seagull category,JLL Auto Parts,Search-result snippet,observed_in_search_snippet_page_not_opened; number-to-part mapping not observed (inferred match ...,False,Potentially distinct source families — independence not established
8,BYD Seagull,Shock absorbers,EQEA2905100A,REC-0036,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (n...,REC-0039,Landin Parts (landinparts.com),Landin,Fetched Landin catalogue listing,observed_directly_in_fetched_listing_text; compatibility 


Exported:
/content/ev_parts_project/output/EV_Parts_Crosscheck_Summary.csv
/content/ev_parts_project/output/EV_Parts_Crosscheck_Candidate_Matches.csv


In [ ]:
# PART 10.2 — Manual Cross-Check Review Worksheet

review_candidates = [
    {
        "vehicle": "BYD Seagull",
        "part_category": "Cabin air filters",
        "matching_identifier": "1476508400",
        "record_ids": "REC-0051, REC-0190, REC-0191",
        "initial_observation": "Same identifier across an aggregator, AliExpress and Amazon search results.",
        "main_concern": "All are snippets; identifier inferred from aggregator title."
    },
    {
        "vehicle": "BYD Seagull",
        "part_category": "Cabin air filters",
        "matching_identifier": "EQEA8100242",
        "record_ids": "REC-0190, REC-0191",
        "initial_observation": "Same OE identifier appears in AliExpress and Amazon search results.",
        "main_concern": "Neither individual product page was opened."
    },
    {
        "vehicle": "BYD Seagull",
        "part_category": "Headlights",
        "matching_identifier": "EQEA4121010B",
        "record_ids": "REC-0009, REC-0196",
        "initial_observation": "Landin and JLL evidence contain the same OE identifier.",
        "main_concern": "JLL number-to-product mapping is inferred."
    },
    {
        "vehicle": "BYD Yuan Up",
        "part_category": "Tail lights",
        "matching_identifier": "SC3E4107300",
        "record_ids": "REC-0155, REC-0156",
        "initial_observation": "Same identifier appears in two Alibaba supplier records.",
        "main_concern": "Search snippets; exact product mapping and seller independence need checking."
    },
    {
        "vehicle": "BYD Yuan Up",
        "part_category": "Tail lights",
        "matching_identifier": "1463391400",
        "record_ids": "REC-0156, REC-0157",
        "initial_observation": "Same identifier appears in two Alibaba supplier records.",
        "main_concern": "Search snippets; product specification and compatibility need checking."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Brake pads",
        "matching_identifier": "0446578020",
        "record_ids": "REC-0055, REC-0058, REC-0060, REC-0061, REC-0062",
        "initial_observation": "Same identifier appears across three classified source families.",
        "main_concern": "Retailer relationship and conflicting vehicle fitment details."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Brake pads",
        "matching_identifier": "04466F4020",
        "record_ids": "REC-0056, REC-0057, REC-0059",
        "initial_observation": "Same identifier appears in multiple ToyotaPartsNow/ToyotaPartsDeal records.",
        "main_concern": "All records belong to the same identified source family."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Bumpers",
        "matching_identifier": "521194A957",
        "record_ids": "REC-0110, REC-0111",
        "initial_observation": "Same identifier appears in ToyotaPartsNow and North Park Toyota evidence.",
        "main_concern": "Exact bumper configuration and fitment need checking."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Cabin air filters",
        "matching_identifier": "8713942040",
        "record_ids": "REC-0073, REC-0075, REC-0078",
        "initial_observation": "Same identifier appears across Toyota retailer and Conquest Auto records.",
        "main_concern": "Confirm product description and exact vehicle applicability."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Cabin air filters",
        "matching_identifier": "8713942050",
        "record_ids": "REC-0074, REC-0076, REC-0077",
        "initial_observation": "Same identifier appears across Toyota retailer and Nucar Toyota records.",
        "main_concern": "Distinguish this filter variant from 87139-42040."
    },
    {
        "vehicle": "Toyota bZ4X",
        "part_category": "Wiper blades",
        "matching_identifier": "8521452110",
        "record_ids": "REC-0086, REC-0087",
        "initial_observation": "Same identifier appears in ToyotaPartsDeal and Nunawading Toyota evidence.",
        "main_concern": "Driver-side and right-hand descriptions may conflict."
    }
]

manual_review = pd.DataFrame(review_candidates)

# Add fields for our manual assessment.
manual_review["source_independence"] = "Pending"
manual_review["product_mapping"] = "Pending"
manual_review["vehicle_compatibility"] = "Pending"
manual_review["final_classification"] = "Pending review"
manual_review["review_notes"] = ""

# Display worksheet.
print("MANUAL CROSS-CHECK REVIEW WORKSHEET")
print("Candidate identifier combinations:", len(manual_review))

display(manual_review)

# Export worksheet.
review_path = OUTPUT_DIR / "EV_Parts_Manual_Crosscheck_Review.csv"
manual_review.to_csv(review_path, index=False)

print("\nExported:", review_path)

MANUAL CROSS-CHECK REVIEW WORKSHEET
Candidate identifier combinations: 11


,vehicle,part_category,matching_identifier,record_ids,initial_observation,main_concern,source_independence,product_mapping,vehicle_compatibility,final_classification,review_notes
0,BYD Seagull,Cabin air filters,1476508400,"REC-0051, REC-0190, REC-0191","Same identifier across an aggregator, AliExpress and Amazon search results.",All are snippets; identifier inferred from aggregator title.,Pending,Pending,Pending,Pending review,
1,BYD Seagull,Cabin air filters,EQEA8100242,"REC-0190, REC-0191",Same OE identifier appears in AliExpress and Amazon search results.,Neither individual product page was opened.,Pending,Pending,Pending,Pending review,
2,BYD Seagull,Headlights,EQEA4121010B,"REC-0009, REC-0196",Landin and JLL evidence contain the same OE identifier.,JLL number-to-product mapping is inferred.,Pending,Pending,Pending,Pending review,
3,BYD Yuan Up,Tail lights,SC3E4107300,"REC-0155, REC-0156",Same identifier appears in two Alibaba supplier records.,Search snippets; exact product mapping and seller independence need checking.,Pending,Pending,Pending,Pending review,
4,BYD Yuan Up,Tail lights,1463391400,"REC-0156, REC-0157",Same identifier appears in two Alibaba supplier records.,Search snippets; product specification and compatibility need checking.,Pending,Pending,Pending,Pending review,
5,Toyota bZ4X,Brake pads,0446578020,"REC-0055, REC-0058, REC-0060, REC-0061, REC-0062",Same identifier appears across three classified source families.,Retailer relationship and conflicting vehicle fitment details.,Pending,Pending,Pending,Pending review,
6,Toyota bZ4X,Brake pads,04466F4020,"REC-0056, REC-0057, REC-0059",Same identifier appears in multiple ToyotaPartsNow/ToyotaPartsDeal records.,All records belong to the same identified source family.,Pending,Pending,Pending,Pending review,
7,Toyota bZ4X,Bumpers,521194A957,"REC-0110, REC-0111",Same identifier appears in ToyotaPartsNow and North Park Toyota evidence.,Exact bumper configuration and fitment need checking.,Pending,Pending,Pending,Pending review,
8,Toyota bZ4X,Cabin air filters,8713942040,"REC-0073, REC-0075, REC-0078",Same identifier appears across Toyota retailer and Conquest Auto records.,Confirm product description and exact vehicle applicability.,Pending,Pending,Pending,Pending review,
9,Toyota bZ4X,Cabin air filters,8713942050,"REC-0074, REC-0076, REC-0077",Same identifier appears across Toyota retailer and Nucar Toyota records.,Distinguish this filter variant from 87139-42040.,Pending,Pending,Pending,Pending review,



Exported: /content/ev_parts_project/output/EV_Parts_Manual_Crosscheck_Review.csv


## 11. Evidence quality and limitations

Create transparent review flags. These are **not scores** and are not grounds for deleting a row.

Evidence can be:
- model-named / direct or fetched information;
- search-result snippet only;
- title-level or multi-model claim;
- supplier capability statement;
- compatibility indicated but not confirmed;
- conflicting or uncertain.

The research notes describe important limitations: 12/39 covered combinations have weak-only evidence; Phase 2 records are search excerpts; supplier/operator independence is limited; and no part-level Ethiopian price/stock listing was found for the selected models.

In [ ]:
# Simple field-level review flags.
raw["flag_missing_part_number"] = raw["part_number"].str.strip().eq("")
raw["flag_missing_oe_number"] = raw["oe_part_number"].str.strip().eq("")
raw["flag_missing_compatibility_evidence"] = raw["compatibility_evidence"].str.strip().eq("")
raw["flag_missing_price"] = raw["price"].str.strip().eq("")
raw["flag_missing_availability"] = raw["availability_status"].str.strip().eq("")
raw["flag_missing_source_url"] = raw["source_url"].str.strip().eq("")
raw["flag_compatibility_review"] = raw["compatibility_status"].str.lower().str.contains(
    "weak|uncertain|unconfirmed|multi|title|snippet|not confirmed|conflict",regex=True,na=False
)

flags = [c for c in raw.columns if c.startswith("flag_")]
display(raw[flags].sum().rename("records_flagged").to_frame())

# Source/evidence types as collected; inspect before any grouping decisions.
display(raw["evidence_type"].value_counts(dropna=False).to_frame("records"))
display(evidence["verification_status"].value_counts(dropna=False).to_frame("records"))

,records_flagged
flag_missing_part_number,64
flag_missing_oe_number,75
flag_missing_compatibility_evidence,0
flag_missing_price,72
flag_missing_availability,125
flag_missing_source_url,0
flag_compatibility_review,59


,records
evidence_type,
product listing (price listing),65
catalogue/product listing (price listing where price shown),41
product listing / catalogue entry,16
product listing title,8
product listing title (title names the category),8
supplier listing with category attributes,8
product listing,7
supplier capability statement (not an individual product listing),7
supplier capability statement,5


,records
verification_status,
observed_in_search_snippet_page_not_opened,137
observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (not independently verified),34
observed_directly_in_fetched_page_text,8
observed_directly_in_fetched_listing_text; compatibility = supplier model-selector label only (not independently verified); category_mapping_inferred,7
observed_directly_in_fetched_page_text; category_mapping_inferred,4
observed_in_search_snippet_page_not_opened; Amharic copied exactly as returned (no normalisation),2
observed_in_search_snippet_page_not_opened; part_number_field_inferred_from_title_suffix,1
observed_in_search_snippet_page_not_opened; supplier_attribution_not_verified,1
observed_in_search_snippet_page_not_opened; Amharic text copied exactly as returned by the search tool,1


## 12. Cleaning and standardization

Cleaning is performed on a copy and is conservative:
- preserve every original column/value with a `_raw` suffix;
- trim leading/trailing whitespace;
- normalize Unicode text to NFC (canonical Unicode normalization);
- standardize vehicle/category labels only using explicit mappings;
- parse prices only when a numeric amount can be safely extracted, retaining original price and currency;
- do not infer missing stock, compatibility, price, or demand;
- do not merge listings merely because they look similar.

### Amharic handling
Original Amharic text is preserved without translation or spelling correction.
Unicode NFC normalization is applied to standardize canonically equivalent Unicode representations.


In [ ]:
# Keep an untouched backup of the working raw values in *_raw columns.
for col in raw_original.columns:
    raw[f"{col}_raw"] = raw_original[col]

# Strip whitespace and NFC-normalize text in working columns only.
for col in raw_original.columns:
    raw[col] = raw[col].astype(str).str.strip().map(
        lambda x: unicodedata.normalize("NFC", x)
    )

# Explicit normalization maps; unknown labels remain as collected.
vehicle_map = {
    "BYD Seagull":"BYD Seagull",
    "BYD Yuan Up":"BYD Yuan Up",
    "Changan BenBen E-Star":"Changan BenBen E-Star",
    "Changan Benben E-Star":"Changan BenBen E-Star",
    "Toyota bZ4X":"Toyota bZ4X"
}
category_map = {
    "Brake pads":"Brake pads", "Brake discs":"Brake discs",
    "Cabin air filters":"Cabin air filters", "Cabin-air filters":"Cabin air filters",
    "Wiper blades":"Wiper blades", "Headlights":"Headlights",
    "Tail lights":"Tail lights", "Side mirrors":"Side mirrors",
    "Bumpers":"Bumpers", "Fenders/body panels":"Fenders/body panels",
    "Shock absorbers":"Shock absorbers"
}
raw["vehicle_clean"] = raw["vehicle"].replace(vehicle_map)
raw["part_category_clean"] = raw["part_category"].replace(category_map)

# Source-type normalization: conservative lowercase/trim only.
raw["source_type_clean"] = raw["source_type"].str.strip().str.lower()

display(raw[["vehicle","vehicle_clean"]].drop_duplicates())
display(raw[["part_category","part_category_clean"]].drop_duplicates())

,vehicle,vehicle_clean
0,BYD Seagull,BYD Seagull
49,BYD Yuan Up,BYD Yuan Up
53,Changan BenBen E-Star,Changan BenBen E-Star
54,Toyota bZ4X,Toyota bZ4X
138,General (not model-specific),General (not model-specific)


,part_category,part_category_clean
0,Brake pads,Brake pads
4,Brake discs,Brake discs
6,Wiper blades,Wiper blades
8,Headlights,Headlights
11,Tail lights,Tail lights
14,Side mirrors,Side mirrors
16,Bumpers,Bumpers
25,Fenders/body panels,Fenders/body panels
35,Shock absorbers,Shock absorbers
50,Cabin air filters,Cabin air filters


## 13. Price parsing (without currency conversion)

Prices may be exact, ranges, text, or price-on-request. We create a numeric helper only when a numeric token exists; the original price and currency remain intact. This is not an ETB landed-cost calculation.

In [ ]:
# ============================================================
# PART 13 — PRICE PARSING, CURRENCY STANDARDIZATION & ETB CONVERSION
# ============================================================

import re
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Common reference exchange rates
# Source: National Bank of Ethiopia (NBE)
# Reference date: 2026-11-16
# ------------------------------------------------------------

EXCHANGE_RATE_DATE = "2026-11-16"

EXCHANGE_RATES_TO_ETB = {
    "USD": 160.9987,
    "CNY": 24.2494,
    "CAD": 113.5305
}

# ------------------------------------------------------------
# 2. Standardize currency labels
# ------------------------------------------------------------

def standardize_currency(value):
    currency = str(value).strip().upper()

    if currency in ["RMB", "CNY"]:
        return "CNY"

    if currency == "CAD":
        return "CAD"

    if currency.startswith("USD"):
        return "USD"

    if currency in ["", "NAN", "NONE"]:
        return "UNKNOWN"

    return "UNKNOWN"


raw["currency_clean"] = raw["currency"].map(standardize_currency)

# ------------------------------------------------------------
# 3. Extract numerical values from original price expressions
# ------------------------------------------------------------

def extract_price_numbers(value):
    if not str(value).strip():
        return []

    s = str(value).replace(",", "")
    numbers = re.findall(r"\d+(?:\.\d+)?", s)

    return [float(n) for n in numbers]


raw["price_numbers"] = raw["price"].map(extract_price_numbers)

# Preserve compatibility with the previous notebook
raw["price_numeric_first"] = raw["price_numbers"].apply(
    lambda x: x[0] if len(x) > 0 else np.nan
)

# ------------------------------------------------------------
# 4. Interpret price expressions
# ------------------------------------------------------------

def classify_price(value, numbers):
    s = str(value).strip().lower()

    if not s:
        return "missing"

    if "request" in s:
        return "price_on_request"

    if len(numbers) == 0:
        return "unparseable"

    if len(numbers) >= 2:
        return "range_or_multiple_values_review"

    return "single_numeric_value"


raw["price_type"] = raw.apply(
    lambda row: classify_price(row["price"], row["price_numbers"]),
    axis=1
)

raw["price_parse_note"] = raw["price_type"]

# ------------------------------------------------------------
# 5. Create price bounds
# ------------------------------------------------------------

raw["price_numeric_low"] = raw["price_numbers"].apply(
    lambda x: min(x) if len(x) == 1 else np.nan
)

raw["price_numeric_high"] = raw["price_numbers"].apply(
    lambda x: max(x) if len(x) == 1 else np.nan
)

# Multiple-number expressions require manual review.

# ------------------------------------------------------------
# 6. Attach exchange rates
# ------------------------------------------------------------

raw["exchange_rate_to_etb"] = raw["currency_clean"].map(
    EXCHANGE_RATES_TO_ETB
)

raw["exchange_rate_date"] = EXCHANGE_RATE_DATE

# ------------------------------------------------------------
# 7. Convert prices into Ethiopian Birr
# ------------------------------------------------------------

raw["price_etb"] = (
    raw["price_numeric_first"] *
    raw["exchange_rate_to_etb"]
)

# Only convert single numerical prices with known currencies
raw.loc[
    raw["price_type"] != "single_numeric_value",
    "price_etb"
] = np.nan

# Round converted prices to two decimal places
raw["price_etb"] = raw["price_etb"].round(2)

# ------------------------------------------------------------
# 8. Conversion status
# ------------------------------------------------------------

def conversion_status(row):

    if row["price_type"] == "missing":
        return "missing_price"

    if row["price_type"] == "price_on_request":
        return "price_on_request"

    if row["price_type"] == "unparseable":
        return "unparseable_price"

    if row["currency_clean"] == "UNKNOWN":
        return "unknown_currency"

    if row["price_type"] != "single_numeric_value":
        return "manual_price_review"

    return "converted"


raw["conversion_status"] = raw.apply(
    conversion_status,
    axis=1
)

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

display(
    raw[
        [
            "price",
            "currency",
            "currency_clean",
            "price_numeric_first",
            "price_type",
            "exchange_rate_to_etb",
            "price_etb",
            "conversion_status"
        ]
    ].head(25)
)

,price,currency,currency_clean,price_numeric_first,price_type,exchange_rate_to_etb,price_etb,conversion_status
0,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
1,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
2,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
3,¥64,RMB,CNY,64.0,single_numeric_value,24.2494,1551.96,converted
4,¥108,RMB,CNY,108.0,single_numeric_value,24.2494,2618.94,converted
5,¥94,RMB,CNY,94.0,single_numeric_value,24.2494,2279.44,converted
6,Price on request,,UNKNOWN,NaN,price_on_request,NaN,NaN,price_on_request
7,Price on request,,UNKNOWN,NaN,price_on_request,NaN,NaN,price_on_request
8,¥418,RMB,CNY,418.0,single_numeric_value,24.2494,10136.25,converted
9,¥418,RMB,CNY,418.0,single_numeric_value,24.2494,10136.25,converted


In [ ]:
# Currency conversion summary

print("Currency distribution:")
display(raw["currency_clean"].value_counts(dropna=False))

print("\nConversion status:")
display(raw["conversion_status"].value_counts(dropna=False))

print("\nConverted price statistics:")
display(raw.loc[raw["conversion_status"] == "converted", "price_etb"].describe())

print("\nUnknown currency records:")
display(
    raw.loc[
        raw["conversion_status"] == "unknown_currency",
        ["record_id", "vehicle", "part_category", "price", "currency"]
    ]
)

Currency distribution:


,count
currency_clean,
UNKNOWN,84
USD,82
CNY,29
CAD,2



Conversion status:


,count
conversion_status,
converted,97
missing_price,72
manual_price_review,16
price_on_request,12



Converted price statistics:


,price_etb
count,97.000000
mean,19029.856392
std,30891.912353
min,1246.130000
25%,4001.150000
50%,9336.020000
75%,22979.340000
max,177066.370000



Unknown currency records:


,record_id,vehicle,part_category,price,currency


In [ ]:
# Inspect possible measurement information in text fields

measurement_terms = (
    r"\b\d+(?:\.\d+)?\s*"
    r"(?:mm|cm|m|inches|inch|in|kg|g|lbs?|"
    r"oz|pcs|pieces|sets|pairs)\b"
)

for col in [
    "compatibility_evidence",
    "part_number",
    "oe_part_number"
]:
    matches = raw[
        raw[col].str.contains(
            measurement_terms,
            case=False,
            regex=True,
            na=False
        )
    ]

    print(f"\n{col}: {len(matches)} possible measurement records")

    display(
        matches[["record_id", col]].head(20)
    )


compatibility_evidence: 7 possible measurement records


,record_id,compatibility_evidence
123,REC-0124,Vehicle/fitment text observed: Listed on '2023 Toyota bZ4X Shock Absorber' page | Notes: Other n...
148,REC-0149,"Listing title (Chinese): '万孚36028刹车片后片适配比亚迪秦EV/秦PLUSDM-i/元UP/海豚' (rear pads, number 36028 in tit..."
164,REC-0165,Listing names 'BYD Atto 2 (3/2025+)' not 'Yuan Up'. Wikipedia states the Atto 2 is also called B...
174,REC-0175,Title: 'Applicable to the Front Bumper of Changan Benben E-STAR Car Body Kit'; price '$22 - 26.9...
180,REC-0181,Excerpt: 'China Changan Auto Parts New Benben E-Star LED Rear Tail Light Red and Clear Lens for ...
190,REC-0191,Titles seen: 'Compatible With B-YD SEAGULL SEA MEW EV LAK2059 CABIN FILTER 14765084-00 EQEA-8100...
191,REC-0192,Title: 'Cabin Air Filter Compatible For BYD Seagull EV 1303865000 | Compatible For BYD Seagull E...



part_number: 0 possible measurement records


,record_id,part_number



oe_part_number: 0 possible measurement records


,record_id,oe_part_number


## 14. Duplicate analysis

We distinguish:
- **Exact duplicate rows:** every field identical.
- **Potential same listing:** same source URL/product identifier or same vehicle/category/part number.
- **Different listings for the same part:** potentially useful price/supplier evidence, not automatically duplicates.
- **Variant differences:** may represent real fitment distinctions and must not be collapsed without evidence.

This section creates reports only; it does not remove records.

In [ ]:
# Find records with unknown currency AND a numerical price

unknown_priced = raw[
    (raw["currency_clean"] == "UNKNOWN") &
    (raw["price_type"] == "single_numeric_value")
].copy()

print("Unknown-currency records with numerical prices:",
      len(unknown_priced))

display(
    unknown_priced[
        [
            "record_id",
            "vehicle",
            "part_category",
            "source_name",
            "price",
            "currency",
            "source_url"
        ]
    ].to_string(index=False)
)

Unknown-currency records with numerical prices: 0


'Empty DataFrame\nColumns: [record_id, vehicle, part_category, source_name, price, currency, source_url]\nIndex: []'

In [ ]:
exact_dupes = raw_original[raw_original.duplicated(keep=False)].copy()
potential_key = ["vehicle_clean","part_category_clean","part_number","oe_part_number"]
potential_dupes = raw[raw.duplicated(subset=potential_key,keep=False) & raw["part_number"].ne("")].copy()

url_dupes = raw[raw.duplicated(subset=["source_url","vehicle_clean","part_category_clean"],keep=False) & raw["source_url"].ne("")].copy()

print("Exact duplicate raw rows:",len(exact_dupes))
print("Potential part-number matches:",len(potential_dupes))
print("Same URL/model/category groups:",len(url_dupes))
display(potential_dupes.sort_values(potential_key).head(40))

Exact duplicate raw rows: 0
Potential part-number matches: 30
Same URL/model/category groups: 64


,record_id,vehicle,manufacturer,part_category,source_name,source_type,source_url,collection_date,evidence_type,compatibility_status,...,part_number_raw,price_raw,currency_raw,availability_status_raw,oe_part_number_raw,vehicle_clean,part_category_clean,source_type_clean,price_numeric_first,price_parse_note
62,REC-0063,Toyota bZ4X,Toyota (genuine part number as displayed),Brake discs,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,43512-78020,$88.39,USD,Ships in 1-3 Business Days,43512-78020,Toyota bZ4X,Brake discs,genuine toyota parts online retailer (us-orien...,88.39,first numeric token extracted; inspect ranges
64,REC-0065,Toyota bZ4X,Toyota (genuine part number as displayed),Brake discs,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsdeal.com/oem-2023-toyota-bz...,2026-11-15,product listing / catalogue entry,supplier_stated_fitment_page_level_only,...,43512-78020,,,,43512-78020,Toyota bZ4X,Brake discs,genuine toyota parts online retailer (us-orien...,NaN,missing
54,REC-0055,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,04465-78020,$91.83,USD,Ships in 1-3 Business Days,04465-78020,Toyota bZ4X,Brake pads,genuine toyota parts online retailer (us-orien...,91.83,first numeric token extracted; inspect ranges
57,REC-0058,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsdeal.com/oem-2024-toyot...,2026-11-15,product listing / catalogue entry,supplier_stated_fitment,...,04465-78020,,,,04465-78020,Toyota bZ4X,Brake pads,genuine toyota parts online retailer (us-orien...,NaN,missing
59,REC-0060,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsdeal.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsdeal.com/oem-toyota-bz4x-br...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,04465-78020,$91.44,USD,,04465-78020,Toyota bZ4X,Brake pads,genuine toyota parts online retailer (us-orien...,91.44,first numeric token extracted; inspect ranges
60,REC-0061,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,directfactoryparts.com,genuine OEM parts online retailer (US-oriented...,https://www.directfactoryparts.com/oem-parts/t...,2026-11-15,product listing (price listing),supplier_stated_fitment_conflicting,...,04465-78020,$92.99,USD,,04465-78020,Toyota bZ4X,Brake pads,genuine oem parts online retailer (us-oriented...,92.99,first numeric token extracted; inspect ranges
61,REC-0062,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,neozeroauto.com,Toyota/Lexus parts specialist catalogue page (...,https://neozeroauto.com/part/04465-78020,2026-11-15,product listing / catalogue entry,supplier_stated_fitment_conflicting,...,04465-78020,,,,04465-78020,Toyota bZ4X,Brake pads,toyota/lexus parts specialist catalogue page (...,NaN,missing
55,REC-0056,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://www.toyotapartsnow.com/parts-select/20...,2026-11-15,product listing (price listing),supplier_stated_fitment_page_level_only,...,04466-F4020,$55.09,USD,Ships in 1-3 Business Days,04466-F4020,Toyota bZ4X,Brake pads,genuine toyota parts online retailer (us-orien...,55.09,first numeric token extracted; inspect ranges
56,REC-0057,Toyota bZ4X,Toyota (genuine part number as displayed),Brake pads,toyotapartsnow.com,genuine Toyota parts online retailer (US-orien...,https://toyotapartsnow.com/parts-select/toyota...,2026-11-15,product listing (price listing),supplier_stated_fitment,...,04466-F402

## 15. Compatibility review

Compatibility is not assumed from a part name alone. The source's claim and the strength of evidence must remain visible. The raw compatibility status and evidence are retained; review flags do not upgrade or downgrade claims automatically.

In [ ]:
compatibility_review = raw[[
    "record_id","vehicle_clean","part_category_clean","source_name","source_url",
    "compatibility_status","compatibility_evidence","part_number","oe_part_number",
    "flag_compatibility_review"
]].copy()
display(compatibility_review[compatibility_review["flag_compatibility_review"]].head(100))

,record_id,vehicle_clean,part_category_clean,source_name,source_url,compatibility_status,compatibility_evidence,part_number,oe_part_number,flag_compatibility_review
41,REC-0042,BYD Seagull,Brake pads,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page text: 'We support all the vehicle parts of BYD Seagull, such as: doors, bumpers, lights, ho...",,,True
42,REC-0043,BYD Seagull,Shock absorbers,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page text: 'We support all the vehicle parts of BYD Seagull, such as: doors, bumpers, lights, ho...",,,True
43,REC-0044,BYD Seagull,Bumpers,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page text: 'We support all the vehicle parts of BYD Seagull, such as: doors, bumpers, lights, ho...",,,True
44,REC-0045,BYD Seagull,Brake discs,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page's general inventory paragraph (not Seagull-specific) lists 'Braking system: brake discs, br...",,,True
45,REC-0046,BYD Seagull,Headlights,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page's general inventory paragraph (not Seagull-specific) lists body parts 'headlights, rear tai...",,,True
46,REC-0047,BYD Seagull,Tail lights,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page's general inventory paragraph (not Seagull-specific) lists body parts 'headlights, rear tai...",,,True
47,REC-0048,BYD Seagull,Fenders/body panels,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,unconfirmed_supplier_general_claim,"Page's general inventory paragraph (not Seagull-specific) lists body parts 'hood, front bumper, ...",,,True
48,REC-0049,BYD Seagull,Brake pads,byd-accessories.pages.dev ('Best BYD spare parts' aggregator),https://byd-accessories.pages.dev/byd-spare-parts-best/,unconfirmed_multi_model_title,Listing title: 'Rear Brake Pad Set For BYD Seagull Dolphin Seal King Chazor Song Plus Han Qin Ta...,,,True
49,REC-0050,BYD Yuan Up,Brake pads,byd-accessories.pages.dev ('Best BYD spare parts' aggregator),https://byd-accessories.pages.dev/byd-spare-parts-best/,unconfirmed_multi_model_title,SAME underlying listing as the Seagull rear brake pad record: title includes 'Yuan Up' among man...,,,True
50,REC-0051,BYD Seagull,Cabin air filters,byd-accessories.pages.dev ('Best BYD spare parts' aggregator),https://byd-accessories.pages.dev/byd-spare-parts-best/,unconfirmed_listing_title_only,Listing title: 'Cabin A/C Fresh Air Filter For BYD Seagull Accessories 2023 2024 2025 EV SEA MEW...,1476508400,,True


## 16. Cleaning validation and summary

Compare row counts before/after, check original fields remain, and review standardized labels. Cleaning should not silently drop observations.

In [ ]:
print("Original raw rows:",len(raw_original))
print("Working rows:",len(raw))
print("Rows removed during cleaning:",len(raw_original)-len(raw))
print("Original columns preserved with *_raw:",all(f"{c}_raw" in raw.columns for c in raw_original.columns))

print("\nVehicle summary:")
display(raw[~raw["part_category_clean"].str.upper().str.startswith("N/A")].groupby("vehicle_clean").agg(
    records=("record_id","count"),categories=("part_category_clean","nunique")
).reset_index())

print("\nCategory summary:")
display(raw[~raw["part_category_clean"].str.upper().str.startswith("N/A")].groupby(
    "part_category_clean").size().rename("records").reset_index())

Original raw rows: 197
Working rows: 197
Rows removed during cleaning: 0
Original columns preserved with *_raw: True

Vehicle summary:


,vehicle_clean,records,categories
0,BYD Seagull,59,10
1,BYD Yuan Up,20,9
2,Changan BenBen E-Star,25,10
3,General (not model-specific),1,1
4,Toyota bZ4X,77,10



Category summary:


,part_category_clean,records
0,Brake discs,14
1,Brake pads,24
2,Brake pads (general market context; not EV-spe...,1
3,Bumpers,28
4,Cabin air filters,14
5,Fenders/body panels,18
6,Headlights,19
7,Shock absorbers,19
8,Side mirrors,15
9,Tail lights,14


## 17. Export cleaned working files

The original dataset remains unchanged. These files are explicitly named **Working** and retain original raw fields plus cleaning helpers.

In [ ]:
cleaned_out = OUTPUT_DIR / "EV_Parts_Cleaned_Working.csv"
evidence_out = OUTPUT_DIR / "EV_Parts_Evidence_Log_Working.csv"
coverage_out = OUTPUT_DIR / "EV_Parts_Coverage_Matrix_Working.csv"
quality_out = OUTPUT_DIR / "EV_Parts_Data_Quality_Flags.csv"
duplicates_out = OUTPUT_DIR / "EV_Parts_Potential_Duplicates.csv"
compat_out = OUTPUT_DIR / "EV_Parts_Compatibility_Review.csv"

raw.to_csv(cleaned_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
evidence.to_csv(evidence_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
coverage.to_csv(coverage_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
raw[["record_id"]+flags].to_csv(quality_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
potential_dupes.to_csv(duplicates_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)
compatibility_review.to_csv(compat_out,index=False,encoding="utf-8",quoting=csv.QUOTE_ALL)

print("Exported:")
for p in [cleaned_out,evidence_out,coverage_out,quality_out,duplicates_out,compat_out]:
    print(p)

Exported:
/content/ev_parts_project/output/EV_Parts_Cleaned_Working.csv
/content/ev_parts_project/output/EV_Parts_Evidence_Log_Working.csv
/content/ev_parts_project/output/EV_Parts_Coverage_Matrix_Working.csv
/content/ev_parts_project/output/EV_Parts_Data_Quality_Flags.csv
/content/ev_parts_project/output/EV_Parts_Potential_Duplicates.csv
/content/ev_parts_project/output/EV_Parts_Compatibility_Review.csv


In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/ev_parts_project/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /content/ev_parts_project/output


In [ ]:
import os

output_dir = "/content/ev_parts_project/output"

print("Files currently in output folder:")
for f in sorted(os.listdir(output_dir)):
    print(f)

Files currently in output folder:
EV_Parts_Cleaned_Working.csv
EV_Parts_Compatibility_Review.csv
EV_Parts_Coverage_Matrix_Working.csv
EV_Parts_Crosscheck_Candidate_Matches.csv
EV_Parts_Crosscheck_Summary.csv
EV_Parts_Data_Quality_Flags.csv
EV_Parts_Evidence_Log_Working.csv
EV_Parts_Manual_Crosscheck_Review.csv
EV_Parts_Potential_Duplicates.csv


## Part 18: Designing the Scoring Methodology

### 18.1
Our methodology uses evidence from the collected dataset to evaluate each
vehicle–part combination.

### 18.2 Proposed Evaluation Factors

We propose four main factors for determining inventory priority:

**1. Market Relevance**

Evaluates the available evidence indicating whether a vehicle and its
replacement parts are relevant to the Ethiopian market.

Evidence may include vehicle-market observations, local marketplace
listings, and other relevant market indicators.

These indicators are proxies for market relevance, not confirmed sales
or actual replacement-part demand.

**2. Local Availability Gap**

Evaluates whether there is evidence that a replacement part is difficult
to obtain locally.

Limited local listings or availability information may indicate a possible
supply gap. However, a lack of online listings alone does not prove that
a part is unavailable in Ethiopia.

**3. Compatibility Confidence**

Evaluates how strongly the collected evidence supports the compatibility
of a replacement part with the specified vehicle.

Relevant evidence includes vehicle-specific catalogue entries, part numbers,
OE references, and supplier fitment information.

A compatibility claim from a supplier is not automatically considered
independently verified.

**4. Sourcing Practicality**

Evaluates how practical it appears to source the replacement part.

Relevant evidence includes identified suppliers, product listings,
part identifiers, prices, and availability information.

Missing information about shipping, minimum order quantities, delivery time,
and import costs will be acknowledged as a limitation.

### 18.3 Separate Evidence Confidence Assessment

Evidence confidence will be assessed separately from inventory priority.

This distinction is important because missing evidence does not necessarily
mean that a part has low demand or low importance.

For example, the absence of a BYD Yuan Up brake-disc listing in our
investigated sources does not prove that the part is unnecessary or
unavailable.

Instead, it indicates that our current evidence is insufficient.

Therefore, evidence confidence will help us communicate the reliability
of the resulting scores without automatically treating missing information
as poor performance.

### 18.4 Scoring Unit

The primary scoring unit will be a vehicle–part combination.

With four vehicles and ten replacement-part categories, there are
40 possible combinations.

Each combination will be evaluated using the available evidence.
The results will then be aggregated to compare the ten part categories
and identify the five recommended categories.

### 18.5 Methodological Principle

The scoring method will be evidence-based and transparent.

We will not invent demand figures, failure rates, local stock levels,
or compatibility claims when the collected sources do not establish them.

The scoring scales, factor weights, missing-evidence treatment, and
aggregation procedure will be defined in the following sections.

In [ ]:
import pandas as pd

# Part 18: Proposed scoring framework

scoring_factors = pd.DataFrame([
    {
        "factor": "Market Relevance",
        "role": "Priority factor",
        "purpose": "Evaluate evidence of the vehicle's relevance to the Ethiopian market.",
        "supporting_evidence": (
            "Vehicle-market observations, local marketplace listings, "
            "and relevant market indicators"
        )
    },
    {
        "factor": "Local Availability Gap",
        "role": "Priority factor",
        "purpose": "Evaluate evidence of possible difficulty obtaining the part locally.",
        "supporting_evidence": (
            "Local listings, availability information, and observed supply gaps"
        )
    },
    {
        "factor": "Compatibility Confidence",
        "role": "Priority factor",
        "purpose": "Evaluate how strongly the evidence supports vehicle-part compatibility.",
        "supporting_evidence": (
            "Vehicle-specific catalogue entries, part numbers, OE references, "
            "and supplier fitment claims"
        )
    },
    {
        "factor": "Sourcing Practicality",
        "role": "Priority factor",
        "purpose": "Evaluate the practical possibility of sourcing the part.",
        "supporting_evidence": (
            "Supplier identification, product listings, prices, "
            "and availability information"
        )
    },
    {
        "factor": "Evidence Confidence",
        "role": "Separate reliability assessment",
        "purpose": "Communicate the strength and reliability of the evidence supporting the assessment.",
        "supporting_evidence": (
            "Source quality, verification status, corroboration, "
            "and completeness of supporting information"
        )
    }
])

display(scoring_factors)

,factor,role,purpose,supporting_evidence
0,Market Relevance,Priority factor,Evaluate evidence of the vehicle's relevance to the Ethiopian market.,"Vehicle-market observations, local marketplace listings, and relevant market indicators"
1,Local Availability Gap,Priority factor,Evaluate evidence of possible difficulty obtaining the part locally.,"Local listings, availability information, and observed supply gaps"
2,Compatibility Confidence,Priority factor,Evaluate how strongly the evidence supports vehicle-part compatibility.,"Vehicle-specific catalogue entries, part numbers, OE references, and supplier fitment claims"
3,Sourcing Practicality,Priority factor,Evaluate the practical possibility of sourcing the part.,"Supplier identification, product listings, prices, and availability information"
4,Evidence Confidence,Separate reliability assessment,Communicate the strength and reliability of the evidence supporting the assessment.,"Source quality, verification status, corroboration, and completeness of supporting information"


## Part 19: Scoring Framework

We will evaluate each of the 40 vehicle–part combinations using five factors:

| Factor | What we evaluate |
|---|---|
| Market Relevance | Relevance of the vehicle to the Ethiopian market. |
| Local Availability Gap | Evidence that the part is difficult to obtain locally. |
| Compatibility Confidence | Strength of the vehicle–part compatibility evidence. |
| Sourcing Practicality | Practicality of sourcing the part from suppliers. |
| Evidence Confidence | Reliability and strength of the supporting evidence. |

**Scoring approach:** Each factor will be scored on a scale of 1–5. The five scores will be combined using a weighted formula to calculate the final priority score.

The weights will reflect the relative importance of each factor in our research. Missing information will be treated as unknown rather than automatically assigned the lowest score.

In [ ]:
# Part 19: Define the scoring scale

scoring_scale = pd.DataFrame({
    "score": [1, 2, 3, 4, 5],
    "interpretation": [
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
})

display(scoring_scale)

,score,interpretation
0,1,Very Low
1,2,Low
2,3,Moderate
3,4,High
4,5,Very High


### 19.1 Detailed Scoring Rubric

The following criteria guide how scores are assigned consistently across all vehicle–part combinations.

Scores must be supported by recorded evidence. Insufficient information will be marked **N/A** rather than guessed.

In [ ]:
# Part 19.3: Detailed scoring rubric

scoring_rubric = {
    "Market Relevance": {
        1: "Very little evidence of vehicle presence in Ethiopia.",
        2: "Limited evidence from one weak source.",
        3: "Some credible evidence of local vehicle presence.",
        4: "Multiple credible indicators of market presence.",
        5: "Strong, consistent evidence from multiple sources."
    },

    "Local Availability Gap": {
        1: "Strong evidence that the part is readily available locally.",
        2: "Some evidence of local availability.",
        3: "Limited or inconsistent local availability evidence.",
        4: "Strong evidence of limited local availability.",
        5: "Multiple verified sources demonstrate a significant local supply gap."
    },

    "Compatibility Confidence": {
        1: "Generic listing with no clear vehicle compatibility.",
        2: "Ambiguous or multi-model fitment claim.",
        3: "Supplier-stated vehicle-specific fitment.",
        4: "Vehicle-specific catalogue entry with part identifier.",
        5: "Independently corroborated or officially verified fitment."
    },

    "Sourcing Practicality": {
        1: "No identifiable sourcing option.",
        2: "General supplier identified, but no specific product listing.",
        3: "Specific product listing and supplier identified.",
        4: "Product listing with price or useful availability information.",
        5: "Strong sourcing information, including confirmed stock and practical ordering details."
    },

    "Evidence Confidence": {
        1: "Weak, incomplete or ambiguous evidence.",
        2: "Single search-result snippet or poorly verified claim.",
        3: "Relevant source page or listing observed.",
        4: "Detailed evidence with strong source relevance.",
        5: "Reliable, independently corroborated or officially verified evidence."
    }
}

rubric_rows = []

for factor, levels in scoring_rubric.items():
    for score, description in levels.items():
        rubric_rows.append({
            "factor": factor,
            "score": score,
            "interpretation": scoring_scale.loc[
                scoring_scale["score"] == score, "interpretation"
            ].iloc[0],
            "criteria": description
        })

scoring_rubric_df = pd.DataFrame(rubric_rows)

display(scoring_rubric_df)

,factor,score,interpretation,criteria
0,Market Relevance,1,Very Low,Very little evidence of vehicle presence in Ethiopia.
1,Market Relevance,2,Low,Limited evidence from one weak source.
2,Market Relevance,3,Moderate,Some credible evidence of local vehicle presence.
3,Market Relevance,4,High,Multiple credible indicators of market presence.
4,Market Relevance,5,Very High,"Strong, consistent evidence from multiple sources."
5,Local Availability Gap,1,Very Low,Strong evidence that the part is readily available locally.
6,Local Availability Gap,2,Low,Some evidence of local availability.
7,Local Availability Gap,3,Moderate,Limited or inconsistent local availability evidence.
8,Local Availability Gap,4,High,Strong evidence of limited local availability.
9,Local Availability Gap,5,Very High,Multiple verified sources demonstrate a significant local supply gap.


## 19.2 Weight Assignment

The weights reflect the relative importance of each factor in determining import priority.

| Factor | Weight |
|---|---:|
| Market Relevance | 35% |
| Compatibility Confidence | 25% |
| Local Availability Gap | 20% |
| Sourcing Practicality | 15% |
| Evidence Confidence | 5% |
| **Total** | **100%** |

**Rationale:** Market relevance receives the highest weight because the objective is to identify parts associated with vehicles that matter in the Ethiopian market. Compatibility receives the second-highest weight because incorrect fitment creates financial and operational risks. Local availability and sourcing practicality determine whether an import opportunity is worthwhile and feasible. Evidence confidence receives a smaller weight because it evaluates the reliability of supporting information rather than the market opportunity itself.

In [ ]:
# Part 19.2: Define and validate factor weights

factor_weights = {
    "Market Relevance": 0.35,
    "Compatibility Confidence": 0.25,
    "Local Availability Gap": 0.20,
    "Sourcing Practicality": 0.15,
    "Evidence Confidence": 0.05
}

weights_df = pd.DataFrame([
    {"factor": factor, "weight": weight}
    for factor, weight in factor_weights.items()
])

weights_df["weight_percentage"] = weights_df["weight"] * 100

display(weights_df[["factor", "weight_percentage"]])

total_weight = sum(factor_weights.values())

print(f"Total weight: {total_weight * 100:.0f}%")

assert abs(total_weight - 1.0) < 1e-9, "Weights must add up to 100%."

print("Weight validation passed.")

,factor,weight_percentage
0,Market Relevance,35.0
1,Compatibility Confidence,25.0
2,Local Availability Gap,20.0
3,Sourcing Practicality,15.0
4,Evidence Confidence,5.0


Total weight: 100%
Weight validation passed.


Part 20: Applying the Scoring Method


In [ ]:
# Part 20.1: Build detailed evidence review for all 40 combinations

import pandas as pd

# Start with the evaluation table
evidence_review = evaluation[
    ["vehicle", "part_category", "record_ids"]
].copy()

# Expand the record IDs into individual rows
evidence_review["record_id"] = evidence_review["record_ids"].apply(
    lambda x: str(x).split("; ") if pd.notna(x) else []
)

evidence_review = evidence_review.drop(columns=["record_ids"])
evidence_review = evidence_review.explode("record_id")

evidence_review["record_id"] = evidence_review["record_id"].astype(str).str.strip()

evidence_review = evidence_review[
    evidence_review["record_id"].ne("")
    & evidence_review["record_id"].ne("nan")
]

# Exclude columns already present to prevent duplicate-column conflicts
raw_details = raw.drop(
    columns=[
        col for col in raw.columns
        if col in evidence_review.columns and col != "record_id"
    ]
)

# Attach the original evidence details
evidence_review = evidence_review.merge(
    raw_details,
    on="record_id",
    how="left",
    validate="many_to_one"
)

# Select available evidence fields for inspection
preferred_fields = [
    "record_id",
    "vehicle",
    "part_category",
    "source_name",
    "source_url",
    "source_type",
    "title",
    "description",
    "part_number",
    "oe_number",
    "compatibility_status",
    "evidence_status",
    "price",
    "currency",
    "availability",
    "notes",
    "collection_date"
]

display_fields = [
    col for col in preferred_fields
    if col in evidence_review.columns
]

print("Total linked evidence records:", len(evidence_review))
print("Evidence fields available:", display_fields)

display(evidence_review[display_fields])

# Export the evidence review
evidence_review.to_csv(
    OUTPUT_DIR / "EV_Parts_Detailed_Evidence_Review.csv",
    index=False
)

print("\nDetailed evidence review exported successfully.")

Total linked evidence records: 181
Evidence fields available: ['record_id', 'vehicle', 'part_category', 'source_name', 'source_url', 'source_type', 'part_number', 'compatibility_status', 'price', 'currency', 'collection_date']


,record_id,vehicle,part_category,source_name,source_url,source_type,part_number,compatibility_status,price,currency,collection_date
0,REC-0001,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd0042260f1,aftermarket body/parts supplier catalogue (China),BYD0042260F1,supplier_stated_fitment_unverified,¥64,RMB,2026-11-15
1,REC-0002,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd0042260r1,aftermarket body/parts supplier catalogue (China),BYD0042260R1,supplier_stated_fitment_unverified,¥64,RMB,2026-11-15
2,REC-0003,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd0042260f1p,aftermarket body/parts supplier catalogue (China),BYD0042260F1P,supplier_stated_fitment_unverified,¥64,RMB,2026-11-15
3,REC-0004,BYD Seagull,Brake pads,Landin Parts (landinparts.com),https://www.landinparts.com/p.html#/byd/byd0042260r1p,aftermarket body/parts supplier catalogue (China),BYD0042260R1P,supplier_stated_fitment_unverified,¥64,RMB,2026-11-15
4,REC-0042,BYD Seagull,Brake pads,CQSW / Chongqing Shiwei Technology (cqswautoparts.com),https://www.cqswautoparts.com/wholesale-superior-accessories-byd-seagull-2023-2024-2025-auto-par...,wholesale trading exporter website (China),,unconfirmed_supplier_general_claim,,,2026-11-15
...,...,...,...,...,...,...,...,...,...,...,...
176,REC-0127,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyota-bz4x-shock_absorber.html,genuine Toyota parts online retailer (US-oriented; USD),48520-80742,supplier_stated_fitment,$128.69,USD,2026-11-15
177,REC-0128,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyota-bz4x-shock_absorber.html,genuine Toyota parts online retailer (US-oriented; USD),48510-80C35,supplier_stated_fitment,$129.16,USD,2026-11-15
178,REC-0129,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyota-bz4x-shock_absorber.html,genuine Toyota parts online retailer (US-oriented; USD),48520-80785,supplier_stated_fitment,$124.15,USD,2026-11-15
179,REC-0130,Toyota bZ4X,Shock absorbers,toyotapartsdeal.com,https://www.toyotapartsdeal.com/oem-2023-toyota-bz4x-shock_absorber.html,genuine Toyota parts online retailer (US-oriented; USD),48520-80738,supplier_stated_fitment,$129.86,USD,2026-11-15



Detailed evidence review exported successfully.


In [ ]:
# Part 20.2: Examine contextual market evidence

part_record_ids = set(evidence_review["record_id"])

context_records = raw[
    ~raw["record_id"].isin(part_record_ids)
].copy()

print("Total contextual records:", len(context_records))
print("\nAvailable columns:")
print(context_records.columns.tolist())

display(context_records)

print("\nContextual records by source:")
if "source_name" in context_records.columns:
    display(
        context_records.groupby("source_name")
        .size()
        .reset_index(name="record_count")
        .sort_values("record_count", ascending=False)
    )

Total contextual records: 16

Available columns:
['record_id', 'vehicle', 'manufacturer', 'part_category', 'source_name', 'source_type', 'source_url', 'collection_date', 'evidence_type', 'compatibility_status', 'compatibility_evidence', 'part_number', 'price', 'currency', 'availability_status', 'oe_part_number', 'flag_missing_part_number', 'flag_missing_oe_number', 'flag_missing_compatibility_evidence', 'flag_missing_price', 'flag_missing_availability', 'flag_missing_source_url', 'flag_compatibility_review', 'record_id_raw', 'vehicle_raw', 'manufacturer_raw', 'part_category_raw', 'source_name_raw', 'source_type_raw', 'source_url_raw', 'collection_date_raw', 'evidence_type_raw', 'compatibility_status_raw', 'compatibility_evidence_raw', 'part_number_raw', 'price_raw', 'currency_raw', 'availability_status_raw', 'oe_part_number_raw', 'vehicle_clean', 'part_category_clean', 'source_type_clean', 'price_numeric_first', 'price_parse_note', 'currency_clean', 'price_numbers', 'price_type', 'pric

,record_id,vehicle,manufacturer,part_category,source_name,source_type,source_url,collection_date,evidence_type,compatibility_status,...,price_parse_note,currency_clean,price_numbers,price_type,price_numeric_low,price_numeric_high,exchange_rate_to_etb,exchange_rate_date,price_etb,conversion_status
131,REC-0132,BYD Seagull,BYD,N/A (vehicle presence evidence - not a parts record),Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/byd/seagull,2026-11-15,market observation (vehicle presence),not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
132,REC-0133,BYD Yuan Up,BYD,N/A (vehicle presence evidence - not a parts record),Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/,2026-11-15,market observation (vehicle presence),not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
133,REC-0134,Changan BenBen E-Star,Changan,N/A (vehicle presence evidence - not a parts record),Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/changan,2026-11-15,market observation (vehicle presence),not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
134,REC-0135,Toyota bZ4X,Toyota,N/A (vehicle presence evidence - not a parts record),Gezat Electric (ev.gezat.et),Ethiopian EV discovery/marketplace website,https://ev.gezat.et/toyota,2026-11-15,market observation (vehicle presence),not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
135,REC-0136,BYD Yuan Up,BYD,N/A (vehicle presence evidence - not a parts record),Hulu Market (hulumarket.com.et),Ethiopian classifieds marketplace,https://hulumarket.com.et/car/byd-yuan-plus,2026-11-15,"vehicle sales listing (market observation, Amharic/English mixed)",not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
136,REC-0137,Toyota bZ4X,Toyota,N/A (vehicle presence evidence - not a parts record),Beki Autocar (bekiautocar.com),Ethiopian car broker/dealer website,https://bekiautocar.com/,2026-11-15,"vehicle sales broker statement (market observation, English/Amharic mixed)",not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
137,REC-0138,BYD Seagull,BYD,N/A (vehicle presence evidence - not a parts record),Beki Autocar (bekiautocar.com),Ethiopian car broker/dealer website,https://bekiautocar.com/,2026-11-15,"vehicle sales broker statement (market observation, English/Amharic mixed)",not_applicable_vehicle_presence,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
138,REC-0139,General (not model-specific),Not applicable,N/A (market context - not one of the ten categories),2merkato.com business directory - Ayu Begemdir Auto Parts (Ayenew Mersha),Ethiopian business directory entry (local EV spare-parts supplier),https://www.2merkato.com/directory/41387-አዩ-በጌምድር-የመኪና-ዕቃ-መለዋወጫ,2026-11-15,"market observation (local supplier self-description, Amharic + English)",not_applicable_market_context,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
139,REC-0140,General (not model-specific),Not applicable,N/A (market context - not one of the ten categories),Addis Fortune - MOENCO introduces BYD Shark 6,Ethiopian news outlet,https://addisfortune.news/moenco-introduces-byd-shark-6-as-hybrid-demand-gains-ground,2026-11-15,market observation (news),not_applicable_market_context,...,missing,UNKNOWN,[],missing,NaN,NaN,NaN,2026-11-16,NaN,missing_price
140,REC-0141,General (not model-specific),Not applicable,N/A (market context - not one of the ten categories),Mobility Rising newsletter - BYD signs partnership with Inchcape,industry newsletter,https://mobility-rising.beehiiv.com/p/byd-signs-partnership-with-inchcape-to-enter-ethiopia,2026-11-15,market observation (news),not_applic


Contextual records by source:


,source_name,record_count
8,Gezat Electric (ev.gezat.et),4
4,Beki Autocar (bekiautocar.com),2
0,2merkato.com business directory - Ayu Begemdir Auto Parts (Ayenew Mersha),1
1,Addis Fortune - MOENCO introduces BYD Shark 6,1
3,Adey Market (adey-market.com),1
2,Addis Fortune - spare-parts price article (Toyota/general),1
6,China-Global South Project - Ethiopia's middle class ditching gasoline vehicles,1
5,CQSW case study - Ethiopian auto parts distributor,1
7,Ethiopanorama (ethiopanorama.com) article,1
9,Hulu Market (hulumarket.com.et),1


In [ ]:
# PART 20.4 — Assign Scores to the Five Evaluation Factors

import numpy as np
import pandas as pd

# 1. Market Relevance
market_scores = {
    "BYD Seagull": 4,
    "BYD Yuan Up": 4,
    "Changan BenBen E-Star": 3,
    "Toyota bZ4X": 4
}

# 2. Compatibility Confidence
def score_compatibility(status):
    if pd.isna(status):
        return np.nan

    status = str(status).strip().lower()

    if status == "supplier_stated_fitment":
        return 3

    if status in [
        "supplier_stated_fitment_unverified",
        "supplier_stated_fitment_page_level_only",
        "supplier_stated_fitment_title_only"
    ]:
        return 2

    return 1


# 3. Evidence Confidence
def score_evidence_confidence(evidence_type):
    if pd.isna(evidence_type):
        return np.nan

    evidence_type = str(evidence_type).strip().lower()

    if any(term in evidence_type for term in [
        "search", "snippet", "title"
    ]):
        return 2

    if any(term in evidence_type for term in [
        "product listing", "catalogue", "supplier listing"
    ]):
        return 3

    if "supplier capability statement" in evidence_type:
        return 2

    return np.nan


# 4. Sourcing Practicality
def score_sourcing_practicality(row):
    evidence_type = str(row.get("evidence_type", "")).lower()
    price = row.get("price")
    availability = str(row.get("availability_status", "")).strip().lower()

    if "supplier capability statement" in evidence_type:
        return 2

    if any(term in evidence_type for term in [
        "product listing",
        "catalogue",
        "supplier listing"
    ]):
        has_price = pd.notna(price) and str(price).strip().lower() not in [
            "", "nan", "none"
        ]

        has_availability = availability not in [
            "", "nan", "none"
        ]

        return 4 if has_price or has_availability else 3

    return np.nan


# 5. Local Availability Gap estimates
local_gap_estimates = {
    "BYD Seagull": {
        "Brake pads": 3, "Brake discs": 4, "Cabin air filters": 3,
        "Wiper blades": 3, "Headlights": 2, "Tail lights": 3,
        "Side mirrors": 3, "Bumpers": 2, "Fenders/body panels": 2,
        "Shock absorbers": 3
    },
    "BYD Yuan Up": {
        "Brake pads": 3, "Brake discs": 4, "Cabin air filters": 4,
        "Wiper blades": 3, "Headlights": 3, "Tail lights": 3,
        "Side mirrors": 3, "Bumpers": 2, "Fenders/body panels": 3,
        "Shock absorbers": 4
    },
    "Changan BenBen E-Star": {
        "Brake pads": 4, "Brake discs": 5, "Cabin air filters": 4,
        "Wiper blades": 4, "Headlights": 3, "Tail lights": 4,
        "Side mirrors": 4, "Bumpers": 3, "Fenders/body panels": 4,
        "Shock absorbers": 4
    },
    "Toyota bZ4X": {
        "Brake pads": 3, "Brake discs": 3, "Cabin air filters": 3,
        "Wiper blades": 2, "Headlights": 2, "Tail lights": 3,
        "Side mirrors": 3, "Bumpers": 2, "Fenders/body panels": 3,
        "Shock absorbers": 3
    }
}


# Summarize individual evidence records for each combination
evidence_scores = evidence_review.copy()

evidence_scores["compatibility_score"] = (
    evidence_scores["compatibility_status"]
    .apply(score_compatibility)
)

evidence_scores["evidence_score"] = (
    evidence_scores["evidence_type"]
    .apply(score_evidence_confidence)
)

evidence_scores["sourcing_score"] = evidence_scores.apply(
    score_sourcing_practicality,
    axis=1
)

# Use the strongest available documented evidence for each combination
evidence_summary = (
    evidence_scores
    .groupby(["vehicle", "part_category"], as_index=False)
    .agg(
        compatibility_score=("compatibility_score", "max"),
        evidence_score=("evidence_score", "max"),
        sourcing_score=("sourcing_score", "max")
    )
)

# Start with all 40 vehicle-part combinations
scoring_table = evaluation[
    ["vehicle", "part_category", "record_count"]
].copy()

# Add summarized evidence scores
scoring_table = scoring_table.merge(
    evidence_summary,
    on=["vehicle", "part_category"],
    how="left"
)

# Assign the five factor scores
scoring_table["Market Relevance"] = (
    scoring_table["vehicle"].map(market_scores)
)

scoring_table["Compatibility Confidence"] = (
    scoring_table["compatibility_score"]
)

scoring_table["Local Availability Gap"] = scoring_table.apply(
    lambda row: local_gap_estimates.get(
        row["vehicle"], {}
    ).get(row["part_category"], np.nan),
    axis=1
)

scoring_table["Sourcing Practicality"] = (
    scoring_table["sourcing_score"]
)

scoring_table["Evidence Confidence"] = (
    scoring_table["evidence_score"]
)

# Display factor scores only
factor_columns = [
    "Market Relevance",
    "Compatibility Confidence",
    "Local Availability Gap",
    "Sourcing Practicality",
    "Evidence Confidence"
]

display(
    scoring_table[
        ["vehicle", "part_category", "record_count"] + factor_columns
    ]
)

print("Total combinations:", len(scoring_table))
print("Factor scores assigned.")

,vehicle,part_category,record_count,Market Relevance,Compatibility Confidence,Local Availability Gap,Sourcing Practicality,Evidence Confidence
0,BYD Seagull,Brake pads,8,4,2.0,3,4.0,3.0
1,BYD Seagull,Brake discs,3,4,2.0,4,4.0,3.0
2,BYD Seagull,Cabin air filters,6,4,3.0,3,4.0,3.0
3,BYD Seagull,Wiper blades,3,4,2.0,3,4.0,3.0
4,BYD Seagull,Headlights,5,4,3.0,2,4.0,3.0
5,BYD Seagull,Tail lights,4,4,2.0,3,4.0,3.0
6,BYD Seagull,Side mirrors,2,4,2.0,3,4.0,3.0
7,BYD Seagull,Bumpers,10,4,2.0,2,4.0,3.0
8,BYD Seagull,Fenders/body panels,11,4,2.0,2,4.0,3.0
9,BYD Seagull,Shock absorbers,7,4,2.0,3,4.0,3.0


Total combinations: 40
Factor scores assigned.


In [ ]:
# PART 20.5 — Overall Scoring of the 10 Spare-Part Categories

# Agreed scoring weights
weights = {
    "Market Relevance": 0.35,
    "Compatibility Confidence": 0.25,
    "Local Availability Gap": 0.20,
    "Sourcing Practicality": 0.15,
    "Evidence Confidence": 0.05
}

factor_columns = list(weights.keys())

# Keep only combinations with all five factor scores
complete_combinations = scoring_table.dropna(
    subset=factor_columns
).copy()

# Calculate average factor scores for each spare-part category
category_scores = (
    complete_combinations
    .groupby("part_category")
    .agg(
        vehicles_evaluated=("vehicle", "nunique"),
        evidence_records=("record_count", "sum"),
        **{
            factor: (factor, "mean")
            for factor in factor_columns
        }
    )
    .reset_index()
)

# Calculate the overall weighted score for each category
category_scores["Overall Weighted Score"] = sum(
    category_scores[factor] * weight
    for factor, weight in weights.items()
)

# Round scores for presentation
score_columns = factor_columns + ["Overall Weighted Score"]

category_scores[score_columns] = (
    category_scores[score_columns].round(2)
)

# Display the 10 category scores without ranking
display(category_scores)

# Summary
print("Total spare-part categories:", len(category_scores))
print(
    "Categories evaluated across all four vehicles:",
    (category_scores["vehicles_evaluated"] == 4).sum()
)
print(
    "Categories with fewer than four vehicles:",
    (category_scores["vehicles_evaluated"] < 4).sum()
)
print("Weights total:", sum(weights.values()))

# Export category-level scoring results
category_output = (
    "/content/ev_parts_project/output/EV_Parts_Category_Scores_Working.csv"
)

category_scores.to_csv(category_output, index=False)

print("\nCategory scoring results saved to:", category_output)

,part_category,vehicles_evaluated,evidence_records,Market Relevance,Compatibility Confidence,Local Availability Gap,Sourcing Practicality,Evidence Confidence,Overall Weighted Score
0,Brake discs,3,14,3.67,2.00,4.00,3.33,2.67,3.22
1,Brake pads,4,24,3.75,2.75,3.25,4.00,3.00,3.40
2,Bumpers,4,28,3.75,2.50,2.25,4.00,3.00,3.14
3,Cabin air filters,4,14,3.75,2.50,3.50,4.00,3.00,3.39
4,Fenders/body panels,4,18,3.75,2.50,3.00,4.00,2.50,3.26
5,Headlights,4,19,3.75,2.75,2.50,4.00,2.75,3.24
6,Shock absorbers,4,19,3.75,2.25,3.50,4.00,2.75,3.31
7,Side mirrors,4,15,3.75,2.00,3.25,4.00,2.75,3.20
8,Tail lights,4,14,3.75,2.50,3.25,4.00,3.00,3.34
9,Wiper blades,4,16,3.75,2.25,3.00,4.00,3.00,3.22


Total spare-part categories: 10
Categories evaluated across all four vehicles: 9
Categories with fewer than four vehicles: 1
Weights total: 1.0

Category scoring results saved to: /content/ev_parts_project/output/EV_Parts_Category_Scores_Working.csv


In [ ]:
# PART 20.6 — Sensitivity Analysis

# Alternative weighting scenarios
scenarios = {
    "Baseline": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.20,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Market Focus": {
        "Market Relevance": 0.40,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.15,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Compatibility Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.30,
        "Local Availability Gap": 0.20,
        "Sourcing Practicality": 0.10,
        "Evidence Confidence": 0.05
    },

    "Availability Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.20,
        "Local Availability Gap": 0.25,
        "Sourcing Practicality": 0.15,
        "Evidence Confidence": 0.05
    },

    "Sourcing Focus": {
        "Market Relevance": 0.35,
        "Compatibility Confidence": 0.25,
        "Local Availability Gap": 0.15,
        "Sourcing Practicality": 0.20,
        "Evidence Confidence": 0.05
    }
}

# Calculate scores under each scenario
sensitivity_results = category_scores[
    ["part_category", "vehicles_evaluated"]
].copy()

for scenario_name, scenario_weights in scenarios.items():

    sensitivity_results[scenario_name] = sum(
        category_scores[factor] * weight
        for factor, weight in scenario_weights.items()
    )

    # Rank categories within each scenario
    sensitivity_results[f"{scenario_name} Rank"] = (
        sensitivity_results[scenario_name]
        .rank(method="min", ascending=False)
    )

# Count how often each category appears in the top five
rank_columns = [
    f"{scenario_name} Rank"
    for scenario_name in scenarios
]

sensitivity_results["Top Five Frequency"] = (
    sensitivity_results[rank_columns].le(5).sum(axis=1)
)

# Display scores and rankings
display(
    sensitivity_results.sort_values(
        "Baseline",
        ascending=False
    ).round(2)
)

# Display top-five stability summary
stability_summary = (
    sensitivity_results[
        ["part_category", "Top Five Frequency"]
    ]
    .sort_values(
        ["Top Five Frequency", "part_category"],
        ascending=[False, True]
    )
)

print("\nTOP-FIVE STABILITY ACROSS SCENARIOS")
display(stability_summary)

# Export results
sensitivity_output = (
    "/content/ev_parts_project/output/EV_Parts_Sensitivity_Analysis.csv"
)

sensitivity_results.to_csv(sensitivity_output, index=False)

print("\nSensitivity analysis saved to:", sensitivity_output)
print("Number of weighting scenarios:", len(scenarios))

,part_category,vehicles_evaluated,Baseline,Baseline Rank,Market Focus,Market Focus Rank,Compatibility Focus,Compatibility Focus Rank,Availability Focus,Availability Focus Rank,Sourcing Focus,Sourcing Focus Rank,Top Five Frequency
1,Brake pads,4,3.40,1.0,3.42,1.0,3.34,1.0,3.42,2.0,3.44,1.0,5
3,Cabin air filters,4,3.39,2.0,3.40,2.0,3.31,2.0,3.44,1.0,3.41,2.0,5
8,Tail lights,4,3.34,3.0,3.36,3.0,3.26,3.0,3.38,4.0,3.37,3.0,5
6,Shock absorbers,4,3.31,4.0,3.32,4.0,3.22,4.0,3.38,3.0,3.34,4.0,5
4,Fenders/body panels,4,3.26,5.0,3.30,5.0,3.19,5.0,3.29,6.0,3.31,5.0,4
5,Headlights,4,3.24,6.0,3.30,5.0,3.18,6.0,3.22,9.0,3.31,5.0,2
9,Wiper blades,4,3.22,7.0,3.26,7.0,3.14,8.0,3.26,7.0,3.28,7.0,0
0,Brake discs,3,3.22,8.0,3.20,10.0,3.15,7.0,3.32,5.0,3.18,10.0,1
7,Side mirrors,4,3.20,9.0,3.22,8.0,3.10,9.0,3.26,7.0,3.24,8.0,0
2,Bumpers,4,3.14,10.0,3.21,9.0,3.06,10.0,3.12,10.0,3.22,9.0,0



TOP-FIVE STABILITY ACROSS SCENARIOS


,part_category,Top Five Frequency
1,Brake pads,5
3,Cabin air filters,5
6,Shock absorbers,5
8,Tail lights,5
4,Fenders/body panels,4
5,Headlights,2
0,Brake discs,1
2,Bumpers,0
7,Side mirrors,0
9,Wiper blades,0



Sensitivity analysis saved to: /content/ev_parts_project/output/EV_Parts_Sensitivity_Analysis.csv
Number of weighting scenarios: 5


20.5 — Identify the five highest-priority combinations

In [ ]:
# Select the five highest-scoring part categories
final_top_five = (
    category_scores
    .merge(
        sensitivity_results[
            ["part_category", "Top Five Frequency"]
        ],
        on="part_category",
        how="left"
    )
    .sort_values("Overall Weighted Score", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

final_top_five.insert(
    0, "Final Rank", range(1, len(final_top_five) + 1)
)

# Display final recommendations
display(final_top_five[
    [
        "Final Rank",
        "part_category",
        "Overall Weighted Score",
        "vehicles_evaluated",
        "evidence_records",
        "Top Five Frequency"
    ]
])

# Export final recommendations
from pathlib import Path

output_dir = Path(output_dir)

final_top_five.to_csv(
    output_dir / "EV_Parts_Final_Top_Five_Recommendations.csv",
    index=False
)

print("Final top-five recommendations exported successfully.")

,Final Rank,part_category,Overall Weighted Score,vehicles_evaluated,evidence_records,Top Five Frequency
0,1,Brake pads,3.40,4,24,5
1,2,Cabin air filters,3.39,4,14,5
2,3,Tail lights,3.34,4,14,5
3,4,Shock absorbers,3.31,4,19,5
4,5,Fenders/body panels,3.26,4,18,4


Final top-five recommendations exported successfully.


## 21. Interpretation of Final Recommendations

The weighted scoring model identified the following five part categories as the highest-priority candidates for potential import:

1. **Brake pads (3.40):** Ranked first, with evidence across all four selected vehicles and strong sourcing practicality.
2. **Cabin air filters (3.39):** Ranked second, supported by evidence across all four vehicles and strong sourcing practicality.
3. **Tail lights (3.34):** Ranked third, with evidence across all four vehicles and a consistently high score.
4. **Shock absorbers (3.31):** Ranked fourth, supported by evidence across all four vehicles and a relatively high estimated local availability gap.
5. **Fenders/body panels (3.26):** Ranked fifth, with evidence across all four vehicles and a competitive overall score.

These scores represent the results of our proposed prioritization model. They do not directly measure sales volume or guarantee profitability.

The recommendations provide a starting point for deciding which EV spare-part categories deserve further sourcing and market investigation.